# Source-conditioned likelihood feasibility V31
Method: Ko et al., ATLAS, https://arxiv.org/abs/2610.19583. Independent implementation; probe checkpoint must be discarded. No competition submission.

In [ ]:
from pathlib import Path
import subprocess, sys, json
Path('plan.json').write_text("{\n  \"experiment\": \"improvement-v31\",\n  \"kind\": \"two-GPU source-likelihood feasibility\",\n  \"model\": \"qwen-lm/qwen2.5/Transformers/1.5b/1\",\n  \"seed\": 2901,\n  \"sources\": 18,\n  \"rank\": 16,\n  \"alpha\": 32,\n  \"dropout\": 0.05,\n  \"max_length\": 512,\n  \"batch_size\": 4,\n  \"gradient_accumulation\": 4,\n  \"microsteps\": 24,\n  \"head_chunk\": 16,\n  \"train_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\",\n  \"projected_training_microsteps\": 7290,\n  \"pilot_validation_rows\": 540,\n  \"hidden_projection_rows\": 4500,\n  \"other_ensemble_allowance_seconds\": 3600,\n  \"launch_gate_seconds\": 24300,\n  \"paper\": \"https://arxiv.org/abs/2610.19583\",\n  \"authors_repo\": \"https://github.com/komseok0109/ATLAS\",\n  \"reviewed_commit\": \"93c1b15da6143161e222eecee7ebcdc3c60645a9\",\n  \"implementation\": \"Independent PyTorch implementation of published source-specific LoRA and prefix method; no repository code copied\",\n  \"limitations\": [\n    \"Feasibility checkpoint uses long host passages and must be discarded\",\n    \"No accuracy evaluation or selection\",\n    \"1.5B differs from main paper backbone; 18 exact models differ from five families\",\n    \"Full training and hidden runtime projections are estimates from a short probe\",\n    \"Two-GPU data parallelism preserves effective batch32 and total planned training examples; timing includes gradient synchronization\"\n  ],\n  \"world_size\": 2,\n  \"initial_loss_scale\": 128,\n  \"numerical_fix\": \"V1 first update had non-finite gradient norm at loss scale1024; reduce scale128 and record finite gradient diagnostics per microstep\",\n  \"source_sha256\": {\n    \"source_likelihood.py\": \"c816f9f2d9683039bea221cb22520a8daf61959d88a5fec00eba5abbeee7e801\",\n    \"probe_likelihood_v31.py\": \"e42b5da6186a48896b16007bed2ec05fb6f7187fe8fa9a917499a063b976d75a\"\n  }\n}")
Path('source_likelihood.py').write_text('"""Independent implementation of source-conditioned likelihood modeling.\n\nMethod credit: Ko et al., ATLAS, arXiv:2610.19583, equations in sections 3/B.1.\nAuthors\' code reviewed at 93c1b15da6143161e222eecee7ebcdc3c60645a9;\nno source copied. This module uses plain PyTorch adapters, not PEFT internals.\nRouting must stay unchanged until backward completes (including recomputation).\n"""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass SourceRoute:\n    ids: torch.Tensor | None = None\n\n\nclass SourceProjection(nn.Module):\n    def __init__(self, linear, route, sources, rank=16, alpha=32, dropout=.05):\n        super().__init__()\n        self.base = linear.requires_grad_(False)\n        self.route = route\n        self.factor = alpha/rank\n        self.dropout = nn.Dropout(dropout)\n        self.down = nn.Parameter(torch.empty(rank, linear.in_features, device=linear.weight.device))\n        self.up = nn.Parameter(torch.empty(sources, rank, linear.out_features, device=linear.weight.device))\n        nn.init.kaiming_uniform_(self.down, a=math.sqrt(5))\n        nn.init.normal_(self.up, std=1e-4)\n\n    def forward(self, x):\n        ids = self.route.ids\n        if ids is None or ids.shape != (len(x),):\n            raise ValueError(\'Missing or stale source routing\')\n        output = self.base(x)\n        projected = F.linear(self.dropout(x.to(self.down.dtype)), self.down)\n        update = torch.bmm(projected, self.up[ids])\n        return output + (self.factor*update).to(output.dtype)\n\n\ndef clipped_evidence(log_probs, mask, temperatures=(1., 2., 4., 8.)):\n    """[batch, source, token] -> [batch, source]; padding contributes nothing."""\n    if log_probs.ndim != 3 or mask.shape != log_probs[:,0].shape:\n        raise ValueError(\'Invalid token evidence dimensions\')\n    if (mask.sum(-1) == 0).any():\n        raise ValueError(\'Every passage needs a scored token\')\n    lp = log_probs.float()\n    disadvantage = (lp[:,None,:,:]-lp[:,:,None,:]).clamp_min(0).sum(2)\n    evidence = torch.stack([torch.exp(-disadvantage/t) for t in temperatures]).mean(0)\n    return (evidence*mask[:,None,:]).sum(-1)/mask.sum(-1)[:,None]\n\n\nclass SourceLanguageModel(nn.Module):\n    def __init__(self, causal_lm, sources, rank=16, alpha=32, dropout=.05, head_chunk=32):\n        super().__init__()\n        self.lm = causal_lm.requires_grad_(False)\n        self.sources, self.head_chunk = sources, head_chunk\n        self.route = SourceRoute()\n        self.lm.config.use_cache = False\n        names = [name for name, module in self.lm.named_modules()\n                 if isinstance(module, nn.Linear) and name.rsplit(\'.\',1)[-1] in\n                 {\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\'}]\n        if len(names) != 4*self.lm.config.num_hidden_layers:\n            raise ValueError(\'Unexpected attention projection layout\')\n        for name in names:\n            parent_name, attr = name.rsplit(\'.\',1)\n            parent = self.lm.get_submodule(parent_name)\n            setattr(parent, attr, SourceProjection(getattr(parent, attr),self.route,\n                                                  sources,rank,alpha,dropout))\n        embedding = self.lm.get_input_embeddings().weight.detach().float()\n        initial = embedding.mean(0)[None,:].expand(sources,-1).clone()\n        initial += .01*embedding.std()*torch.randn_like(initial)\n        self.prefix = nn.Parameter(initial)\n\n    def token_log_probs(self, token_ids, mask, sources):\n        if sources.shape != (len(token_ids),) or ((sources<0)|(sources>=self.sources)).any():\n            raise ValueError(\'Invalid source IDs\')\n        if token_ids.shape != mask.shape or token_ids.shape[1] < 2:\n            raise ValueError(\'Invalid token batch\')\n        self.route.ids = sources\n        embedding = self.lm.get_input_embeddings()(token_ids)\n        prefix = self.prefix[sources,None,:].to(embedding.dtype)\n        inputs = torch.cat((prefix,embedding),dim=1)\n        attention = torch.cat((torch.ones_like(mask[:,:1]),mask),dim=1)\n        hidden = self.lm.get_decoder()(inputs_embeds=inputs,attention_mask=attention,\n                                      use_cache=False).last_hidden_state[:,1:-1,:]\n        target = token_ids[:,1:]\n\n        def score_chunk(h, target_chunk):\n            logits = self.lm.get_output_embeddings()(h).float()\n            return -F.cross_entropy(logits.flatten(0,1),target_chunk.flatten(),\n                                    reduction=\'none\').view_as(target_chunk)\n\n        parts = []\n        for start in range(0,hidden.shape[1],self.head_chunk):\n            h, target_chunk = hidden[:,start:start+self.head_chunk],target[:,start:start+self.head_chunk]\n            value = checkpoint(score_chunk,h,target_chunk,use_reentrant=False) if self.training else score_chunk(h,target_chunk)\n            parts.append(value)\n        return torch.cat(parts,dim=1)\n\n    def forward(self, token_ids, mask, sources):\n        lp = self.token_log_probs(token_ids,mask,sources)\n        valid = mask[:,1:].float()\n        if (valid.sum(-1)==0).any(): raise ValueError(\'Empty scored passage\')\n        return -((lp*valid).sum(-1)/valid.sum(-1)).mean()\n\n    @torch.no_grad()\n    def all_source_scores(self, token_ids, mask):\n        if self.training: raise ValueError(\'Call eval before scoring all sources\')\n        lp = torch.stack([self.token_log_probs(token_ids,mask,\n                         torch.full((len(token_ids),),k,device=token_ids.device,dtype=torch.long))\n                         for k in range(self.sources)],dim=1)\n        valid = mask[:,1:].float()\n        mean = (lp*valid[:,None,:]).sum(-1)/valid.sum(-1)[:,None]\n        return {\'mean\':mean,\'clipped\':clipped_evidence(lp,valid),\'token_log_probs\':lp}\n\n    def adapter_state(self):\n        return {name:parameter.detach().cpu().clone() for name,parameter in self.named_parameters()\n                if parameter.requires_grad}\n\n    def load_adapter_state(self, state):\n        expected = {name:p for name,p in self.named_parameters() if p.requires_grad}\n        if set(state) != set(expected): raise ValueError(\'Adapter keys differ\')\n        with torch.no_grad():\n            for name, value in state.items():\n                if value.shape != expected[name].shape or not torch.isfinite(value).all():\n                    raise ValueError(\'Invalid adapter tensor: \'+name)\n                expected[name].copy_(value)\n')
Path('probe_likelihood_v31.py').write_text('"""Bounded feasibility measurement; discard this checkpoint after the probe."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nfrom contextlib import nullcontext\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\n\nos.environ[\'HF_HUB_OFFLINE\']=\'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\']=\'1\'\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom safetensors.torch import save_file, load_file\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom source_likelihood import SourceLanguageModel\n\n\ndef main():\n    started=time.monotonic()\n    plan=json.loads(Path(\'plan.json\').read_text())\n    dist.init_process_group(\'nccl\')\n    rank=dist.get_rank()\n    assert dist.get_world_size()==plan[\'world_size\']==2\n    torch.cuda.set_device(rank)\n    def log(**record):\n        record[\'elapsed_seconds\']=time.monotonic()-started\n        with Path(f\'progress-rank{rank}.jsonl\').open(\'a\') as f:f.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record),flush=True)\n    torch.manual_seed(plan[\'seed\'])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\n    candidates=[]\n    for path in Path(\'/kaggle/input\').rglob(\'config.json\'):\n        cfg=json.loads(path.read_text())\n        if cfg.get(\'model_type\')==\'qwen2\' and cfg.get(\'hidden_size\')==1536 and cfg.get(\'num_hidden_layers\')==28:\n            candidates.append(path.parent)\n    assert len(candidates)==1,candidates\n    model_dir=candidates[0]\n    files=list(Path(\'/kaggle/input\').rglob(\'train.csv\'))\n    assert len(files)==1 and hashlib.sha256(files[0].read_bytes()).hexdigest()==plan[\'train_sha256\']\n    data=pd.read_csv(files[0])\n    labels=sorted(data.model.unique())\n    frame=data.assign(length=data.text.str.len()).sort_values([\'length\',\'id\'],ascending=[False,True]).head(32)\n    tok=AutoTokenizer.from_pretrained(model_dir,local_files_only=True)\n    tok.padding_side=\'right\'\n    if tok.pad_token_id is None:tok.pad_token=tok.eos_token\n    texts=tok(frame.text.tolist(),add_special_tokens=False,truncation=True,max_length=plan[\'max_length\']-2)[\'input_ids\']\n    texts=[[tok.eos_token_id]+ids+[tok.eos_token_id] for ids in texts]\n    bank=tok.pad({\'input_ids\':texts},padding=\'max_length\',max_length=plan[\'max_length\'],return_tensors=\'pt\')\n    source=torch.tensor([labels.index(label) for label in frame.model],device=\'cuda\')\n    lm=AutoModelForCausalLM.from_pretrained(model_dir,torch_dtype=torch.float16,local_files_only=True,\n                                          attn_implementation=\'sdpa\').cuda()\n    model=SourceLanguageModel(lm,plan[\'sources\'],plan[\'rank\'],plan[\'alpha\'],plan[\'dropout\'],plan[\'head_chunk\'])\n    model.lm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.train()\n    parallel=DistributedDataParallel(model,device_ids=[rank])\n    params=[p for p in model.parameters() if p.requires_grad]\n    before=model.adapter_state()\n    opt=torch.optim.AdamW(params,lr=1e-4,weight_decay=.01)\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=plan[\'initial_loss_scale\'])\n    opt.zero_grad(set_to_none=True)\n    steps=[]\n    log(event=\'loaded\',trainable_parameters=sum(p.numel() for p in params),torch=torch.__version__,device=torch.cuda.get_device_name())\n    for step in range(plan[\'microsteps\']):\n        begin=time.monotonic()\n        offset=(step*plan[\'world_size\']+rank)*plan[\'batch_size\']\n        rows=torch.arange(offset,offset+plan[\'batch_size\'])%len(frame)\n        ids=bank[\'input_ids\'][rows].cuda();mask=bank[\'attention_mask\'][rows].cuda()\n        update=(step+1)%plan[\'gradient_accumulation\']==0\n        with (nullcontext() if update else parallel.no_sync()):\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                loss=parallel(ids,mask,source[rows.cuda()])/plan[\'gradient_accumulation\']\n            assert torch.isfinite(loss)\n            scaler.scale(loss).backward()\n        nonfinite=[name for name,p in model.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]\n        log(event=\'gradient_diagnostic\',rank=rank,step=step,scale=scaler.get_scale(),nonfinite_parameters=nonfinite)\n        assert not nonfinite,nonfinite\n        if update:\n            scaler.unscale_(opt)\n            norm=torch.nn.utils.clip_grad_norm_(params,1.)\n            assert torch.isfinite(norm)\n            old=scaler.get_scale();scaler.step(opt);scaler.update()\n            assert scaler.get_scale()>=old,\'Skipped optimizer step\'\n            opt.zero_grad(set_to_none=True)\n        torch.cuda.synchronize();seconds=time.monotonic()-begin;steps.append(seconds)\n        log(event=\'microstep\',step=step,loss=float(loss.detach())*plan[\'gradient_accumulation\'],seconds=seconds,update=update)\n    state=model.adapter_state()\n    assert all(torch.isfinite(v).all() for v in state.values())\n    changed=sum(not torch.equal(before[k],v) for k,v in state.items())\n    assert changed==len(state),(changed,len(state))\n    save_file(state,f\'probe-adapter-rank{rank}.safetensors\')\n    model.eval()\n    begin=time.monotonic()\n    with torch.autocast(\'cuda\',dtype=torch.float16):\n        scores=model.all_source_scores(ids,mask)[\'clipped\']\n    torch.cuda.synchronize();score_seconds=time.monotonic()-begin\n    assert torch.isfinite(scores).all()\n    with torch.no_grad():model.prefix.zero_()\n    model.load_adapter_state(load_file(f\'probe-adapter-rank{rank}.safetensors\'))\n    with torch.autocast(\'cuda\',dtype=torch.float16):\n        reloaded=model.all_source_scores(ids,mask)[\'clipped\']\n    torch.testing.assert_close(scores,reloaded,atol=1e-5,rtol=1e-4)\n    per_step=float(np.mean(steps[8:]))\n    pilot=per_step*plan[\'projected_training_microsteps\']+score_seconds*plan[\'pilot_validation_rows\']/plan[\'batch_size\']+600\n    hidden=score_seconds*plan[\'hidden_projection_rows\']/plan[\'batch_size\']+plan[\'other_ensemble_allowance_seconds\']\n    result={\'verified_updates\':True,\'checkpoint_reload_verified\':True,\'changed_tensors\':changed,\n            \'mean_training_microstep_seconds\':per_step,\'all_source_batch_seconds\':score_seconds,\n            \'projected_pilot_seconds\':pilot,\'projected_hidden_seconds\':hidden,\n            \'peak_allocated_bytes\':torch.cuda.max_memory_allocated(),\n            \'qualifies_for_pilot\':pilot<=plan[\'launch_gate_seconds\'] and hidden<=plan[\'launch_gate_seconds\'],\n            \'checkpoint_discard_required\':True,\'submission_authorized\':False}\n    Path(f\'probe-result-rank{rank}.json\').write_text(json.dumps(result,indent=2))\n    log(event=\'complete\',rank=rank,result=result)\n    dist.barrier()\n    dist.destroy_process_group()\n\n\nif __name__==\'__main__\':main()\n')
subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', 'probe_likelihood_v31.py'], check=True)
